In [ ]:
# ── Colab setup: clone repo and install dependencies ──
import os, sys

if not os.path.exists('/content/repo'):
    !git clone https://github.com/joshgreenwa/Graph-Transformer-Attention-Interpretability.git /content/repo

if '/content/repo' not in sys.path:
    sys.path.insert(0, '/content/repo')

os.chdir('/content/repo')

# Match dependency setup used in generate_figures.ipynb
%pip install -r /content/repo/requirements.lock.txt
%pip install -i https://pypi.org/simple rdkit
!pip install numpy==1.26.4
!pip install ogb
!pip install scipy scikit-learn

print('Setup complete')


# Temporary Bias-Routing Expansion Experiments

This notebook is **standalone** and intentionally kept in `tmp_experiments`.

Goals:
1. Reproduce hard top-$k$ vs bottom-$k$ routing scores for reference.
2. Add **soft routing curves** (score as a function of bias percentile threshold).
3. Add **support-vs-weights decomposition** to separate routing (support) from selection (weights).
4. Identify heads that best match the hypothesis: **bias routes, dot-product selects**.


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

from graph_interp.config import set_seed, apply_plot_defaults
from graph_interp.data import load_dataset, load_smiles_pool, build_batch_from_smiles
from graph_interp.extraction import load_model, extract_attention, node_only_conditional
from graph_interp.metrics.scores import sample_transpositions, cossim_lastdim
from graph_interp.metrics.bias_routing import score_bias_routing, score_bias_routing_cossim

apply_plot_defaults()
set_seed(42)

model, device = load_model()
ds = load_dataset()
smiles_pool = load_smiles_pool(ds)
print(f"Loaded model on {device}. SMILES pool: {len(smiles_pool)}")

# ===== Experiment knobs =====
N_HARD = 20
N_SOFT = 20
N_SW = 20

NUM_PERMS_HARD = 32
NUM_PERMS_SOFT = 32
NUM_PERMS_SW = 32

TAU = 0.02
TOP_K_FRAC = 0.50
PERCENTILES = np.arange(5, 101, 5)

SUPPORT_MODE = "topm"    # "topm" or "eps"
SUPPORT_TOPM_FRAC = 0.25
SUPPORT_EPS = 1e-3

EXAMPLE_HEADS = [(1, 24), (11, 14), (0, 0)]


In [ ]:
def _row_topk_mask(scores: torch.Tensor, k: int) -> torch.Tensor:
    """Top-k boolean mask along last dim for any tensor [..., n]."""
    n = scores.shape[-1]
    k = max(1, min(int(k), n))
    _, idx = scores.topk(k, dim=-1)
    mask = torch.zeros_like(scores, dtype=torch.bool)
    mask.scatter_(-1, idx, True)
    return mask


def _ensure_nonempty_mask(mask: torch.Tensor, ref_scores: torch.Tensor) -> torch.Tensor:
    """Guarantee each row has at least one True (fallback to argmax)."""
    empty = ~mask.any(dim=-1)
    if empty.any():
        argmax = ref_scores.argmax(dim=-1, keepdim=True)
        fix = torch.zeros_like(mask, dtype=torch.bool)
        fix.scatter_(-1, argmax, True)
        mask = mask | (fix & empty.unsqueeze(-1))
    return mask


def _renorm_masked(p: torch.Tensor, mask: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    """Mask rows then renormalize each row to sum 1."""
    x = p * mask.to(p.dtype)
    z = x.sum(dim=-1, keepdim=True)
    return x / z.clamp_min(eps)


def _alpha_from_base(A_nodes: torch.Tensor, uv_list: list[tuple[int, int]], tau: float) -> torch.Tensor:
    """Alpha weights used by the key-permutation score, shape [K,H,n_queries]."""
    K = len(uv_list)
    H, n, _ = A_nodes.shape
    delta = torch.empty((K, H, n), device=A_nodes.device, dtype=A_nodes.dtype)
    inv_tau = 1.0 / max(float(tau), 1e-12)
    for t, (u, v) in enumerate(uv_list):
        delta[t] = (A_nodes[:, :, u] - A_nodes[:, :, v]).abs() * inv_tau
    return torch.softmax(delta.float(), dim=0).to(A_nodes.dtype)


@torch.no_grad()
def score_soft_routing_curves_layer(
    A_all: torch.Tensor,
    dot_all: torch.Tensor,
    bias_all: torch.Tensor,
    percentiles: np.ndarray,
    num_perms: int = 32,
    tau: float = 0.02,
    seed: int = 0,
    eps: float = 1e-12,
):
    """Soft routing curves for one layer and all heads.

    For each percentile p, define top-p% keys per (head, query) by bias-only rank,
    then compute conditional structural/symbolic scores and captured attention mass.
    """
    device = A_all.device
    H, T, _ = A_all.shape
    n = T - 1

    P = len(percentiles)
    out = {
        "struct": np.zeros((P, H), dtype=float),
        "sym": np.zeros((P, H), dtype=float),
        "mass": np.zeros((P, H), dtype=float),
    }
    if n < 3:
        return out

    A_nodes = node_only_conditional(A_all, eps=eps)[0].float()  # [H,n,n]
    A_nodes = A_nodes / A_nodes.sum(dim=-1, keepdim=True).clamp_min(eps)
    dot_nodes = dot_all[:, 1:, 1:].float()
    bias_nodes = bias_all[:, 1:, 1:].float()

    B_attn = torch.softmax(bias_nodes, dim=-1)
    B_attn = B_attn / B_attn.sum(dim=-1, keepdim=True).clamp_min(eps)

    masks = []
    mass = torch.zeros((P, H), device=device)
    for pi, pct in enumerate(percentiles):
        k = max(1, int(np.ceil(float(pct) * n / 100.0)))
        m = _row_topk_mask(B_attn, k)
        m = _ensure_nonempty_mask(m, B_attn)
        masks.append(m)
        mass[pi] = (A_nodes * m.float()).sum(dim=-1).mean(dim=-1)

    mask_stack = torch.stack(masks, dim=0)  # [P,H,n,n]

    uv_list = sample_transpositions(n=n, K=num_perms, seed=seed)
    alpha = _alpha_from_base(A_nodes, uv_list, tau=tau)  # [K,H,n]

    struct_acc = torch.zeros((P, H), device=device)
    sym_acc = torch.zeros((P, H), device=device)

    for t, (u, v) in enumerate(uv_list):
        perm = torch.arange(n, device=device)
        tmp = perm[u].clone()
        perm[u] = perm[v]
        perm[v] = tmp

        logits_pi = dot_nodes[:, :, perm] + bias_nodes
        A_pi = torch.softmax(logits_pi, dim=-1)
        A_pi = A_pi / A_pi.sum(dim=-1, keepdim=True).clamp_min(eps)
        A_sym_ref = A_nodes[:, :, perm]

        a_pi_m = _renorm_masked(A_pi.unsqueeze(0), mask_stack, eps=eps)
        a_ref_m = _renorm_masked(A_nodes.unsqueeze(0), mask_stack, eps=eps)
        a_sym_m = _renorm_masked(A_sym_ref.unsqueeze(0), mask_stack, eps=eps)

        cs_struct = cossim_lastdim(a_pi_m, a_ref_m)  # [P,H,n]
        cs_sym = cossim_lastdim(a_pi_m, a_sym_m)

        w = alpha[t].unsqueeze(0)  # [1,H,n]
        struct_acc += (w * cs_struct).mean(dim=-1)
        sym_acc += (w * cs_sym).mean(dim=-1)

    out["struct"] = struct_acc.detach().cpu().numpy()
    out["sym"] = sym_acc.detach().cpu().numpy()
    out["mass"] = mass.detach().cpu().numpy()
    return out


@torch.no_grad()
def compute_soft_curves_one_graph(
    model,
    batch,
    percentiles: np.ndarray,
    num_perms: int = 32,
    tau: float = 0.02,
    seed: int = 0,
):
    """Compute soft routing curves for one graph across all layers."""
    dot_list, bias_list, A_list = extract_attention(model, batch)
    L = len(A_list)
    H = A_list[0].shape[0]
    P = len(percentiles)

    struct = np.zeros((P, L, H), dtype=float)
    sym = np.zeros((P, L, H), dtype=float)
    mass = np.zeros((P, L, H), dtype=float)

    for l in range(L):
        r = score_soft_routing_curves_layer(
            A_all=A_list[l],
            dot_all=dot_list[l],
            bias_all=bias_list[l],
            percentiles=percentiles,
            num_perms=num_perms,
            tau=tau,
            seed=seed + l,
        )
        struct[:, l, :] = r["struct"]
        sym[:, l, :] = r["sym"]
        mass[:, l, :] = r["mass"]

    return {
        "percentiles": np.asarray(percentiles, dtype=float),
        "struct": struct,
        "sym": sym,
        "mass": mass,
    }


@torch.no_grad()
def aggregate_soft_curves(
    model,
    smiles_list,
    device,
    n_graphs: int = 20,
    percentiles: np.ndarray = np.arange(5, 101, 5),
    num_perms: int = 32,
    tau: float = 0.02,
    seed: int = 0,
):
    """Aggregate soft routing curves over graphs.

    Returns means/stds and raw arrays with shape [G,P,L,H].
    """
    graphs = smiles_list[:n_graphs] if n_graphs is not None else smiles_list
    raw_struct, raw_sym, raw_mass = [], [], []
    used = 0

    for gi, smi in enumerate(graphs):
        try:
            batch, n_nodes, dist = build_batch_from_smiles(smi, model=model, device=device)
            if n_nodes < 3:
                continue
            out = compute_soft_curves_one_graph(
                model,
                batch,
                percentiles=percentiles,
                num_perms=num_perms,
                tau=tau,
                seed=seed + 10_000 * gi,
            )
            raw_struct.append(out["struct"])
            raw_sym.append(out["sym"])
            raw_mass.append(out["mass"])
            used += 1
        except Exception:
            continue

    if used == 0:
        raise RuntimeError("No valid graphs processed in aggregate_soft_curves.")

    struct_raw = np.stack(raw_struct, axis=0)
    sym_raw = np.stack(raw_sym, axis=0)
    mass_raw = np.stack(raw_mass, axis=0)

    return {
        "percentiles": np.asarray(percentiles, dtype=float),
        "struct_raw": struct_raw,
        "sym_raw": sym_raw,
        "mass_raw": mass_raw,
        "struct_mean": struct_raw.mean(axis=0),
        "sym_mean": sym_raw.mean(axis=0),
        "mass_mean": mass_raw.mean(axis=0),
        "struct_std": struct_raw.std(axis=0),
        "sym_std": sym_raw.std(axis=0),
        "mass_std": mass_raw.std(axis=0),
        "used_graphs": used,
    }


In [ ]:
def _support_mask_from_probs(
    probs: torch.Tensor,
    mode: str = "topm",
    topm_frac: float = 0.25,
    eps_support: float = 1e-3,
):
    """Construct support mask S_ij from row-distribution probabilities."""
    if mode == "topm":
        n = probs.shape[-1]
        k = max(1, int(np.ceil(float(topm_frac) * n)))
        mask = _row_topk_mask(probs, k)
    elif mode == "eps":
        mask = probs > float(eps_support)
    else:
        raise ValueError(f"Unknown support mode: {mode}")
    mask = _ensure_nonempty_mask(mask, probs)
    return mask


@torch.no_grad()
def score_support_vs_weight_layer(
    A_all: torch.Tensor,
    dot_all: torch.Tensor,
    bias_all: torch.Tensor,
    num_perms: int = 32,
    tau: float = 0.02,
    support_mode: str = "topm",
    support_topm_frac: float = 0.25,
    support_eps: float = 1e-3,
    seed: int = 0,
    eps: float = 1e-12,
):
    """Support-vs-weights decomposition for one layer.

    Support score:
      compares support indicators S under permutation intervention.
    Weight score:
      compares renormalised distributions restricted to support S(ref).
    """
    device = A_all.device
    H, T, _ = A_all.shape
    n = T - 1

    keys = ["support_struct", "support_sym", "weight_struct", "weight_sym"]
    out = {k: np.zeros(H, dtype=float) for k in keys}
    if n < 3:
        return out

    A_nodes = node_only_conditional(A_all, eps=eps)[0].float()  # [H,n,n]
    A_nodes = A_nodes / A_nodes.sum(dim=-1, keepdim=True).clamp_min(eps)
    dot_nodes = dot_all[:, 1:, 1:].float()
    bias_nodes = bias_all[:, 1:, 1:].float()

    S_ref = _support_mask_from_probs(
        A_nodes,
        mode=support_mode,
        topm_frac=support_topm_frac,
        eps_support=support_eps,
    )

    uv_list = sample_transpositions(n=n, K=num_perms, seed=seed)
    alpha = _alpha_from_base(A_nodes, uv_list, tau=tau)  # [K,H,n]

    acc = {k: torch.zeros(H, device=device) for k in keys}

    for t, (u, v) in enumerate(uv_list):
        perm = torch.arange(n, device=device)
        tmp = perm[u].clone()
        perm[u] = perm[v]
        perm[v] = tmp

        logits_pi = dot_nodes[:, :, perm] + bias_nodes
        A_pi = torch.softmax(logits_pi, dim=-1)
        A_pi = A_pi / A_pi.sum(dim=-1, keepdim=True).clamp_min(eps)

        A_sym_ref = A_nodes[:, :, perm]

        S_pi = _support_mask_from_probs(
            A_pi,
            mode=support_mode,
            topm_frac=support_topm_frac,
            eps_support=support_eps,
        )
        S_sym = S_ref[:, :, perm]

        cs_support_struct = cossim_lastdim(S_pi.float(), S_ref.float())
        cs_support_sym = cossim_lastdim(S_pi.float(), S_sym.float())

        w_pi = _renorm_masked(A_pi, S_ref, eps=eps)
        w_ref = _renorm_masked(A_nodes, S_ref, eps=eps)
        w_sym = _renorm_masked(A_sym_ref, S_ref, eps=eps)

        cs_weight_struct = cossim_lastdim(w_pi, w_ref)
        cs_weight_sym = cossim_lastdim(w_pi, w_sym)

        w = alpha[t]
        acc["support_struct"] += (w * cs_support_struct).mean(dim=-1)
        acc["support_sym"] += (w * cs_support_sym).mean(dim=-1)
        acc["weight_struct"] += (w * cs_weight_struct).mean(dim=-1)
        acc["weight_sym"] += (w * cs_weight_sym).mean(dim=-1)

    return {k: v.detach().cpu().numpy() for k, v in acc.items()}


@torch.no_grad()
def compute_support_vs_weight_one_graph(
    model,
    batch,
    num_perms: int = 32,
    tau: float = 0.02,
    support_mode: str = "topm",
    support_topm_frac: float = 0.25,
    support_eps: float = 1e-3,
    seed: int = 0,
):
    """Compute support-vs-weight decomposition across all layers for one graph."""
    dot_list, bias_list, A_list = extract_attention(model, batch)
    L = len(A_list)
    H = A_list[0].shape[0]

    keys = ["support_struct", "support_sym", "weight_struct", "weight_sym"]
    out = {k: np.zeros((L, H), dtype=float) for k in keys}

    for l in range(L):
        r = score_support_vs_weight_layer(
            A_all=A_list[l],
            dot_all=dot_list[l],
            bias_all=bias_list[l],
            num_perms=num_perms,
            tau=tau,
            support_mode=support_mode,
            support_topm_frac=support_topm_frac,
            support_eps=support_eps,
            seed=seed + l,
        )
        for k in keys:
            out[k][l] = r[k]

    return out


@torch.no_grad()
def aggregate_support_vs_weight(
    model,
    smiles_list,
    device,
    n_graphs: int = 20,
    num_perms: int = 32,
    tau: float = 0.02,
    support_mode: str = "topm",
    support_topm_frac: float = 0.25,
    support_eps: float = 1e-3,
    seed: int = 0,
):
    """Aggregate support-vs-weight decomposition over graphs."""
    graphs = smiles_list[:n_graphs] if n_graphs is not None else smiles_list
    keys = ["support_struct", "support_sym", "weight_struct", "weight_sym"]
    raw = {k: [] for k in keys}
    used = 0

    for gi, smi in enumerate(graphs):
        try:
            batch, n_nodes, dist = build_batch_from_smiles(smi, model=model, device=device)
            if n_nodes < 3:
                continue
            r = compute_support_vs_weight_one_graph(
                model,
                batch,
                num_perms=num_perms,
                tau=tau,
                support_mode=support_mode,
                support_topm_frac=support_topm_frac,
                support_eps=support_eps,
                seed=seed + 20_000 * gi,
            )
            for k in keys:
                raw[k].append(r[k])
            used += 1
        except Exception:
            continue

    if used == 0:
        raise RuntimeError("No valid graphs processed in aggregate_support_vs_weight.")

    out = {"used_graphs": used}
    for k in keys:
        arr = np.stack(raw[k], axis=0)  # [G,L,H]
        out[f"{k}_raw"] = arr
        out[f"{k}_mean"] = arr.mean(axis=0)
        out[f"{k}_std"] = arr.std(axis=0)

    return out


In [ ]:
@torch.no_grad()
def compute_hard_routing_one_graph(
    model,
    batch,
    num_perms: int = 32,
    tau: float = 0.02,
    top_k_frac: float = 0.5,
    seed: int = 0,
):
    """Hard top-k / bottom-k routing metrics for one graph."""
    dot_list, bias_list, A_list = extract_attention(model, batch)
    L = len(A_list)
    H = A_list[0].shape[0]

    keys = [
        "containment", "bias_entropy", "full_entropy", "topk_mass_bias", "topk_mass_full",
        "struct_topk", "symb_topk", "struct_botk", "symb_botk", "struct_all", "symb_all",
    ]
    out = {k: np.zeros((L, H), dtype=float) for k in keys}

    for l in range(L):
        r_cont = score_bias_routing(
            A_all=A_list[l],
            dot_all=dot_list[l],
            bias_all=bias_list[l],
            num_perms=num_perms,
            tau=tau,
            top_k_frac=top_k_frac,
            seed=seed + l,
        )
        r_cs = score_bias_routing_cossim(
            A_all=A_list[l],
            dot_all=dot_list[l],
            bias_all=bias_list[l],
            num_perms=num_perms,
            tau=tau,
            top_k_frac=top_k_frac,
            seed=seed + 5000 + l,
        )

        for k in ["containment", "bias_entropy", "full_entropy", "topk_mass_bias", "topk_mass_full"]:
            out[k][l] = r_cont[k]
        for k in ["struct_topk", "symb_topk", "struct_botk", "symb_botk", "struct_all", "symb_all"]:
            out[k][l] = r_cs[k]

    return out


@torch.no_grad()
def aggregate_hard_routing(
    model,
    smiles_list,
    device,
    n_graphs: int = 20,
    num_perms: int = 32,
    tau: float = 0.02,
    top_k_frac: float = 0.5,
    seed: int = 0,
):
    """Aggregate hard routing metrics over graphs."""
    graphs = smiles_list[:n_graphs] if n_graphs is not None else smiles_list

    keys = [
        "containment", "bias_entropy", "full_entropy", "topk_mass_bias", "topk_mass_full",
        "struct_topk", "symb_topk", "struct_botk", "symb_botk", "struct_all", "symb_all",
    ]
    raw = {k: [] for k in keys}
    used = 0

    for gi, smi in enumerate(graphs):
        try:
            batch, n_nodes, dist = build_batch_from_smiles(smi, model=model, device=device)
            if n_nodes < 3:
                continue
            r = compute_hard_routing_one_graph(
                model,
                batch,
                num_perms=num_perms,
                tau=tau,
                top_k_frac=top_k_frac,
                seed=seed + 30_000 * gi,
            )
            for k in keys:
                raw[k].append(r[k])
            used += 1
        except Exception:
            continue

    if used == 0:
        raise RuntimeError("No valid graphs processed in aggregate_hard_routing.")

    out = {"used_graphs": used}
    for k in keys:
        arr = np.stack(raw[k], axis=0)  # [G,L,H]
        out[f"{k}_raw"] = arr
        out[f"{k}_mean"] = arr.mean(axis=0)
        out[f"{k}_std"] = arr.std(axis=0)
    return out


In [ ]:
def _line_with_head_band(ax, x, mat_lh, color, label, marker="o", ls="-"):
    """Plot layer-wise mean across heads with min/max head band."""
    mu = mat_lh.mean(axis=1)
    mn = mat_lh.min(axis=1)
    mx = mat_lh.max(axis=1)
    ax.fill_between(x, mn, mx, color=color, alpha=0.12)
    ax.plot(x, mu, marker=marker, linestyle=ls, linewidth=2.0, color=color, label=label)


def plot_hard_routing_summary(res_hard):
    L = res_hard["struct_all_mean"].shape[0]
    layers = np.arange(L)

    fig, axes = plt.subplots(2, 2, figsize=(14.5, 10.5))

    ax = axes[0, 0]
    _line_with_head_band(ax, layers, res_hard["struct_all_mean"], "#111111", "all keys", marker="o", ls="-")
    _line_with_head_band(ax, layers, res_hard["struct_topk_mean"], "#9b2226", "bias top-k", marker="s", ls="--")
    _line_with_head_band(ax, layers, res_hard["struct_botk_mean"], "#1d3557", "bias bottom-k", marker="^", ls="--")
    ax.set_xlabel(r"Layer index $\ell$")
    ax.set_ylabel("Structural score")
    ax.set_title("Hard routing baseline: structural scores")
    ax.set_xticks(layers)
    ax.grid(True, alpha=0.2)
    ax.legend(fontsize=9)

    ax = axes[0, 1]
    _line_with_head_band(ax, layers, res_hard["symb_all_mean"], "#111111", "all keys", marker="o", ls="-")
    _line_with_head_band(ax, layers, res_hard["symb_topk_mean"], "#9b2226", "bias top-k", marker="s", ls="--")
    _line_with_head_band(ax, layers, res_hard["symb_botk_mean"], "#1d3557", "bias bottom-k", marker="^", ls="--")
    ax.set_xlabel(r"Layer index $\ell$")
    ax.set_ylabel("Symbolic score")
    ax.set_title("Hard routing baseline: symbolic scores")
    ax.set_xticks(layers)
    ax.grid(True, alpha=0.2)
    ax.legend(fontsize=9)

    ax = axes[1, 0]
    sym_gap = res_hard["symb_topk_mean"] - res_hard["symb_botk_mean"]
    struct_gap = res_hard["struct_topk_mean"] - res_hard["struct_botk_mean"]
    _line_with_head_band(ax, layers, sym_gap, "#6a4c93", r"$\Delta$ symbolic (top-k - bottom-k)", marker="o", ls="-")
    _line_with_head_band(ax, layers, struct_gap, "#f9844a", r"$\Delta$ structural (top-k - bottom-k)", marker="s", ls="-")
    ax.axhline(0, color="k", linewidth=0.8)
    ax.set_xlabel(r"Layer index $\ell$")
    ax.set_ylabel("Gap")
    ax.set_title("Gap diagnostics")
    ax.set_xticks(layers)
    ax.grid(True, alpha=0.2)
    ax.legend(fontsize=9)

    ax = axes[1, 1]
    _line_with_head_band(ax, layers, res_hard["containment_mean"], "#2a9d8f", "containment", marker="o", ls="-")
    _line_with_head_band(ax, layers, res_hard["topk_mass_bias_mean"], "#264653", "mass in top-k (bias-only)", marker="s", ls="--")
    _line_with_head_band(ax, layers, res_hard["topk_mass_full_mean"], "#e76f51", "mass in top-k (full attn)", marker="^", ls="--")
    ax.set_xlabel(r"Layer index $\ell$")
    ax.set_ylabel("Fraction")
    ax.set_title("Containment and captured mass")
    ax.set_xticks(layers)
    ax.set_ylim(0.0, 1.0)
    ax.grid(True, alpha=0.2)
    ax.legend(fontsize=9)

    fig.suptitle("Bias-routing baseline (hard top-k/bottom-k)", y=1.01, fontsize=13)
    fig.tight_layout()
    plt.show()


def plot_soft_routing_curves_overall(res_soft, layers_to_plot=(0, 1, 2, 11)):
    pct = res_soft["percentiles"]
    struct = res_soft["struct_mean"]  # [P,L,H]
    sym = res_soft["sym_mean"]
    mass = res_soft["mass_mean"]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))
    layer_colors = plt.cm.viridis(np.linspace(0.08, 0.92, len(layers_to_plot)))

    for color, l in zip(layer_colors, layers_to_plot):
        axes[0].plot(pct, mass[:, l, :].mean(axis=1), "o-", color=color, lw=2, ms=4, label=f"L{l}")
        axes[1].plot(pct, sym[:, l, :].mean(axis=1), "s-", color=color, lw=2, ms=4, label=f"L{l}")
        axes[2].plot(pct, struct[:, l, :].mean(axis=1), "^-", color=color, lw=2, ms=4, label=f"L{l}")

    axes[0].set_title("Captured full-attention mass vs bias percentile")
    axes[0].set_xlabel("Bias percentile threshold")
    axes[0].set_ylabel(r"$m(	au)$")
    axes[0].set_ylim(0, 1.02)

    axes[1].set_title("Symbolic score vs bias percentile")
    axes[1].set_xlabel("Bias percentile threshold")
    axes[1].set_ylabel(r"$s_{sym}(	au)$")
    axes[1].set_ylim(0, 1.02)

    axes[2].set_title("Structural score vs bias percentile")
    axes[2].set_xlabel("Bias percentile threshold")
    axes[2].set_ylabel(r"$s_{struct}(	au)$")
    axes[2].set_ylim(0, 1.02)

    for ax in axes:
        ax.grid(True, alpha=0.2)
        ax.legend(fontsize=9)

    fig.suptitle("Soft routing curves (no single hard-k choice)", y=1.02, fontsize=13)
    fig.tight_layout()
    plt.show()


def plot_soft_routing_for_heads(res_soft, head_pairs):
    pct = res_soft["percentiles"]
    struct = res_soft["struct_mean"]
    sym = res_soft["sym_mean"]
    mass = res_soft["mass_mean"]

    nrows = len(head_pairs)
    fig, axes = plt.subplots(nrows, 3, figsize=(16, 3.7 * nrows), squeeze=False)

    for r, (l, h) in enumerate(head_pairs):
        axes[r, 0].plot(pct, mass[:, l, h], "o-", color="#2a9d8f", lw=2)
        axes[r, 1].plot(pct, sym[:, l, h], "s-", color="#b56576", lw=2)
        axes[r, 2].plot(pct, struct[:, l, h], "^-", color="#355070", lw=2)

        axes[r, 0].set_ylabel(f"L{l}, H{h}")
        axes[r, 0].set_ylim(0, 1.02)
        axes[r, 1].set_ylim(0, 1.02)
        axes[r, 2].set_ylim(0, 1.02)

    titles = [
        "Captured mass $m(	au)$",
        "Symbolic score $s_{sym}(	au)$",
        "Structural score $s_{struct}(	au)$",
    ]
    for c in range(3):
        axes[0, c].set_title(titles[c])
        for r in range(nrows):
            axes[r, c].set_xlabel("Bias percentile threshold")
            axes[r, c].grid(True, alpha=0.2)

    fig.suptitle("Soft routing curves for selected heads", y=1.01, fontsize=13)
    fig.tight_layout()
    plt.show()


def plot_support_weight_decomposition(res_sw):
    mats = {
        "Support structural": res_sw["support_struct_mean"],
        "Support symbolic": res_sw["support_sym_mean"],
        "Support dominance (struct - sym)": res_sw["support_struct_mean"] - res_sw["support_sym_mean"],
        "Weight structural": res_sw["weight_struct_mean"],
        "Weight symbolic": res_sw["weight_sym_mean"],
        "Weight dominance (sym - struct)": res_sw["weight_sym_mean"] - res_sw["weight_struct_mean"],
    }

    fig, axes = plt.subplots(2, 3, figsize=(18, 8.4))
    keys = list(mats.keys())

    for i, name in enumerate(keys):
        ax = axes[i // 3, i % 3]
        mat = mats[name]
        if "dominance" in name:
            im = ax.imshow(mat, aspect="auto", interpolation="nearest", cmap="coolwarm", vmin=-1, vmax=1)
        else:
            im = ax.imshow(mat, aspect="auto", interpolation="nearest", cmap="YlOrRd", vmin=0, vmax=1)
        ax.set_title(name, fontsize=11)
        ax.set_xlabel("Head index $h$")
        ax.set_ylabel(r"Layer index $\ell$")
        ax.set_xticks(range(0, mat.shape[1], 4))
        ax.set_yticks(range(mat.shape[0]))
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)

    fig.suptitle("Support vs weights decomposition", y=1.02, fontsize=13)
    fig.tight_layout()
    plt.show()

    L = res_sw["support_struct_mean"].shape[0]
    layers = np.arange(L)
    fig, ax = plt.subplots(figsize=(10, 4.2))
    ax.plot(layers, (res_sw["support_struct_mean"] - res_sw["support_sym_mean"]).mean(axis=1), "o-", color="#355070", lw=2, label="support dominance (struct-sym)")
    ax.plot(layers, (res_sw["weight_sym_mean"] - res_sw["weight_struct_mean"]).mean(axis=1), "s-", color="#b56576", lw=2, label="weight dominance (sym-struct)")
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xticks(layers)
    ax.set_xlabel(r"Layer index $\ell$")
    ax.set_ylabel("Dominance")
    ax.set_title("Layer-wise decomposition signatures")
    ax.grid(True, alpha=0.2)
    ax.legend(fontsize=9)
    fig.tight_layout()
    plt.show()


def _zscore(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    mu = np.nanmean(x)
    sd = np.nanstd(x)
    if not np.isfinite(sd) or sd < 1e-12:
        return np.zeros_like(x)
    return (x - mu) / sd


def compute_router_head_scores(res_hard, res_soft, res_sw, small_pct=20):
    pct = np.asarray(res_soft["percentiles"], dtype=float)
    idx = int(np.argmin(np.abs(pct - float(small_pct))))

    sym_topk_lift = res_hard["symb_topk_mean"] - res_hard["symb_all_mean"]
    struct_bottom_lift = res_hard["struct_botk_mean"] - res_hard["struct_all_mean"]
    containment = res_hard["containment_mean"]

    mass_small = res_soft["mass_mean"][idx]
    sym_small_gain = res_soft["sym_mean"][idx] - res_soft["sym_mean"][-1]
    struct_tail_gain = res_soft["struct_mean"][-1] - res_soft["struct_mean"][idx]

    support_struct_dom = res_sw["support_struct_mean"] - res_sw["support_sym_mean"]
    weight_sym_dom = res_sw["weight_sym_mean"] - res_sw["weight_struct_mean"]

    parts = {
        "sym_topk_lift": sym_topk_lift,
        "struct_bottom_lift": struct_bottom_lift,
        "containment": containment,
        "mass_small": mass_small,
        "sym_small_gain": sym_small_gain,
        "struct_tail_gain": struct_tail_gain,
        "support_struct_dom": support_struct_dom,
        "weight_sym_dom": weight_sym_dom,
    }

    zsum = np.zeros_like(sym_topk_lift, dtype=float)
    for v in parts.values():
        zsum += _zscore(v)
    composite = zsum / float(len(parts))

    out = dict(parts)
    out["composite"] = composite
    out["small_pct"] = pct[idx]
    return out


def rank_router_heads(router_scores, top_n=20):
    comp = router_scores["composite"]
    L, H = comp.shape
    rows = []
    for l in range(L):
        for h in range(H):
            rows.append((
                float(comp[l, h]),
                l,
                h,
                float(router_scores["sym_topk_lift"][l, h]),
                float(router_scores["struct_bottom_lift"][l, h]),
                float(router_scores["support_struct_dom"][l, h]),
                float(router_scores["weight_sym_dom"][l, h]),
            ))
    rows.sort(key=lambda x: x[0], reverse=True)
    return rows[:top_n]


def plot_router_head_diagnostics(router_scores):
    comp = router_scores["composite"]
    support_dom = router_scores["support_struct_dom"]
    weight_dom = router_scores["weight_sym_dom"]

    L, H = comp.shape
    layers = np.arange(L)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5.3))

    im = axes[0].imshow(comp, aspect="auto", interpolation="nearest", cmap="magma")
    axes[0].set_title("Composite router-selector score")
    axes[0].set_xlabel("Head index $h$")
    axes[0].set_ylabel(r"Layer index $\ell$")
    axes[0].set_xticks(range(0, H, 4))
    axes[0].set_yticks(range(L))
    fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.03)

    x = support_dom.ravel()
    y = weight_dom.ravel()
    c = comp.ravel()
    sc = axes[1].scatter(x, y, c=c, cmap="viridis", s=28, alpha=0.85)
    axes[1].axvline(0, color="k", lw=0.7)
    axes[1].axhline(0, color="k", lw=0.7)
    axes[1].set_xlabel("Support dominance (struct - sym)")
    axes[1].set_ylabel("Weight dominance (sym - struct)")
    axes[1].set_title("Head-level decomposition plane")
    axes[1].grid(True, alpha=0.2)
    fig.colorbar(sc, ax=axes[1], fraction=0.046, pad=0.03, label="Composite")

    thresh = np.percentile(comp, 85)
    candidate = (
        (comp >= thresh)
        & (router_scores["sym_topk_lift"] > 0)
        & (router_scores["struct_bottom_lift"] > 0)
        & (support_dom > 0)
        & (weight_dom > 0)
    )
    counts = candidate.sum(axis=1)
    axes[2].bar(layers, counts, color="#355070", alpha=0.85)
    axes[2].set_xticks(layers)
    axes[2].set_xlabel(r"Layer index $\ell$")
    axes[2].set_ylabel("# candidate heads")
    axes[2].set_title("Candidate heads per layer\n(high composite + all positive signatures)")
    axes[2].grid(True, alpha=0.2, axis="y")

    fig.suptitle("Which heads look like bias routers and dot selectors?", y=1.02, fontsize=13)
    fig.tight_layout()
    plt.show()

    return candidate


## 1) Hard top-k vs bottom-k baseline (reference)


In [ ]:
hard_res = aggregate_hard_routing(
    model=model,
    smiles_list=smiles_pool,
    device=device,
    n_graphs=N_HARD,
    num_perms=NUM_PERMS_HARD,
    tau=TAU,
    top_k_frac=TOP_K_FRAC,
    seed=0,
)
print(f"Hard routing baseline computed on {hard_res['used_graphs']} graphs")
plot_hard_routing_summary(hard_res)


## 2) Soft routing curves (percentile threshold family)


In [ ]:
soft_res = aggregate_soft_curves(
    model=model,
    smiles_list=smiles_pool,
    device=device,
    n_graphs=N_SOFT,
    percentiles=PERCENTILES,
    num_perms=NUM_PERMS_SOFT,
    tau=TAU,
    seed=100,
)
print(f"Soft routing curves computed on {soft_res['used_graphs']} graphs")
plot_soft_routing_curves_overall(soft_res, layers_to_plot=(0, 1, 2, 11))
plot_soft_routing_for_heads(soft_res, EXAMPLE_HEADS)


## 3) Support-vs-weights decomposition


In [ ]:
sw_res = aggregate_support_vs_weight(
    model=model,
    smiles_list=smiles_pool,
    device=device,
    n_graphs=N_SW,
    num_perms=NUM_PERMS_SW,
    tau=TAU,
    support_mode=SUPPORT_MODE,
    support_topm_frac=SUPPORT_TOPM_FRAC,
    support_eps=SUPPORT_EPS,
    seed=200,
)
print(f"Support-vs-weights decomposition computed on {sw_res['used_graphs']} graphs")
plot_support_weight_decomposition(sw_res)


## 4) Identify heads: "bias routes, dot selects"


In [ ]:
router_scores = compute_router_head_scores(hard_res, soft_res, sw_res, small_pct=20)
candidate_mask = plot_router_head_diagnostics(router_scores)

# Top global heads by composite score
rows = rank_router_heads(router_scores, top_n=25)
print(f"Top heads by composite score (small-pct = {router_scores['small_pct']}%)")
print(f"{'rank':>4} {'layer':>5} {'head':>5} {'score':>8} {'symLift':>9} {'strBotLift':>11} {'supDom':>8} {'wDom':>8}")
print("-" * 72)
for i, (score, l, h, sym_lift, str_lift, sup_dom, w_dom) in enumerate(rows, start=1):
    print(f"{i:>4d} {l:>5d} {h:>5d} {score:>8.3f} {sym_lift:>9.3f} {str_lift:>11.3f} {sup_dom:>8.3f} {w_dom:>8.3f}")

# Per-layer candidate head list
L, H = candidate_mask.shape
print("\nCandidate heads per layer:")
for l in range(L):
    hs = np.where(candidate_mask[l])[0].tolist()
    print(f"L{l:>2d}: {hs}")


## 5) Optional robustness checks (parameter sensitivity)

Set `RUN_ROBUSTNESS = True` to run. This is intentionally separate because it is slower.


In [ ]:
RUN_ROBUSTNESS = False

if RUN_ROBUSTNESS:
    settings = [
        {"top_k_frac": 0.40, "support_topm_frac": 0.20},
        {"top_k_frac": 0.50, "support_topm_frac": 0.25},
        {"top_k_frac": 0.60, "support_topm_frac": 0.30},
    ]

    base = None
    base_flat = None
    labels = []
    corr_to_base = []

    for si, st in enumerate(settings):
        print(f"Running robustness setting {si+1}/{len(settings)}: {st}")
        h = aggregate_hard_routing(
            model=model,
            smiles_list=smiles_pool,
            device=device,
            n_graphs=10,
            num_perms=24,
            tau=TAU,
            top_k_frac=st["top_k_frac"],
            seed=10_000 + si,
        )
        sw = aggregate_support_vs_weight(
            model=model,
            smiles_list=smiles_pool,
            device=device,
            n_graphs=10,
            num_perms=24,
            tau=TAU,
            support_mode="topm",
            support_topm_frac=st["support_topm_frac"],
            support_eps=SUPPORT_EPS,
            seed=20_000 + si,
        )
        rs = compute_router_head_scores(h, soft_res, sw, small_pct=20)
        flat = rs["composite"].ravel()

        if base is None:
            base = rs
            base_flat = flat.copy()
            labels.append(f"topk={st['top_k_frac']}, sup={st['support_topm_frac']}")
            corr_to_base.append(1.0)
        else:
            r, _ = spearmanr(base_flat, flat)
            labels.append(f"topk={st['top_k_frac']}, sup={st['support_topm_frac']}")
            corr_to_base.append(float(r))

    fig, ax = plt.subplots(figsize=(8.5, 4.0))
    ax.bar(range(len(labels)), corr_to_base, color="#355070", alpha=0.85)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=20, ha="right")
    ax.set_ylim(0, 1.02)
    ax.set_ylabel("Spearman rank corr vs baseline")
    ax.set_title("Robustness of head ranking across settings")
    ax.grid(True, axis="y", alpha=0.2)
    fig.tight_layout()
    plt.show()
else:
    print("Robustness cell skipped. Set RUN_ROBUSTNESS=True to execute.")
